In [1]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv")["interaction_type"]
y_test = pd.read_csv("../data/processed/y_test.csv")["interaction_type"]

In [3]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "Training Time": training_time
    }

    return results, y_pred

In [5]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg_results, log_reg_pred = evaluate_model(
    log_reg,
    X_train,
    X_test,
    y_train,
    y_test
)

log_reg_results

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'Accuracy': 0.0009881644844700968,
 'Macro Precision': 0.014849603513317622,
 'Macro Recall': 0.039400612020680945,
 'Macro F1': 0.0013360879149172842,
 'Weighted F1': 0.000939727466375247,
 'Training Time': 206.58555912971497}

In [6]:
decision_tree = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

dt_results, dt_pred = evaluate_model(
    decision_tree,
    X_train,
    X_test,
    y_train,
    y_test
)

dt_results

{'Accuracy': 0.47121970938980845,
 'Macro Precision': 0.2570060832891757,
 'Macro Recall': 0.24442355279508965,
 'Macro F1': 0.2468859156850029,
 'Weighted F1': 0.47097480313170415,
 'Training Time': 0.6636970043182373}

In [7]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_results, rf_pred = evaluate_model(
    random_forest,
    X_train,
    X_test,
    y_train,
    y_test
)

rf_results

{'Accuracy': 0.5022345992319267,
 'Macro Precision': 0.28487021143139174,
 'Macro Recall': 0.23450972936211284,
 'Macro F1': 0.2512141550296857,
 'Weighted F1': 0.49402068083990264,
 'Training Time': 5.118413925170898}

In [8]:
results_df = pd.DataFrame([
    {"Model": "Logistic Regression", **log_reg_results},
    {"Model": "Decision Tree", **dt_results},
    {"Model": "Random Forest", **rf_results}
])

results_df

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time
0,Logistic Regression,0.000988,0.014850,0.039401,0.001336,0.000940,206.585559
1,Decision Tree,0.471220,0.257006,0.244424,0.246886,0.470975,0.663697
2,Random Forest,0.502235,0.284870,0.234510,0.251214,0.494021,5.118414


In [9]:
results_df.to_csv("../reports/model_comparison.csv", index=False)

In [11]:
joblib.dump(random_forest, "../models/best_model.pkl")

['../models/best_model.pkl']